# Agri-Intelligence: Fine-Tuning Mistral-7B with QLoRA

In [ ]:
!pip install transformers peft trl accelerate bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 12.1 MB/s eta 0:00:00


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer


### Load Base Model and Configuration & Hyperparameters

In [ ]:

# 🧠 Model to fine-tune
model_name = "mistralai/Mistral-7B-Instruct-v0.3"


dataset_name = "Mahesh2841/Agriculture"

# 🆕 Name for the fine-tuned model
new_model = "mistral-7b-agri-qlora"

#---Lora settings
# LoRA rank
lora_r = 8
# Scaling factor for LoRA layers
lora_alpha = 16
# Dropout in LoRA adapters
lora_dropout = 0.1
#------quantization

# Enable 4-bit loading
use_4bit = True
# Data type used for computations
bnb_4bit_compute_dtype = "float16"
# Quantization type ("nf4" recommended)
bnb_4bit_quant_type = "nf4"
# Use nested quantization (double quant)
use_nested_quant = True

#training arguments
output_dir = "./mistral_agri_results"      # checkpoints & logs
num_train_epochs = 3                       # adjust if needed
fp16 = True                                # Colab GPUs support fp16
bf16 = False                               # bf16 only for A100
per_device_train_batch_size = 1            # small to fit T4/Colab
per_device_eval_batch_size = 1
gradient_accumulation_steps = 4            # effective batch = 8
gradient_checkpointing = True
max_grad_norm = 0.3
learning_rate = 2e-4
weight_decay = 0.001
optim = "paged_adamw_32bit"
lr_scheduler_type = "cosine"
max_steps = -1                             # -1 → use num_train_epochs
warmup_ratio = 0.03
group_by_length = True                     # packs similar-length samples
save_strategy = "epoch"
logging_steps = 50


# 5) SFT / Trainer parameters

max_seq_length = 512          # good default for 7B instruct
packing = False                # True only if data has many shorts
device_map = "auto"            # let HF decide best placement
offload_folder = "./offload"

In [ ]:
!pip install -q huggingface_hub


In [ ]:
from huggingface_hub import login

login(token="TOKEN_")

### Dataset Loading & Preprocessing

In [ ]:
# Load dataset
dataset = load_dataset(dataset_name, split="train")

# Shuffle and select first 1000 rows
dataset = dataset.shuffle(seed=42).select(range(1000))
print(dataset)

# 2) Prepare quantization / BitsAndBytes config
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

# Optional: warn if bf16 is available
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("✅ Your GPU supports bfloat16 – you could set bf16=True for faster training")
        print("=" * 80)


# 3) Load base model in 4-bit

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device_map,
)
model.config.use_cache = False
model.config.pretraining_tp = 1


# 4) Load tokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


# 5) Prepare LoRA (PEFT) config

from peft import LoraConfig
peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
)


# 6) TrainingArguments

from transformers import TrainingArguments

training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_strategy = "steps",        # save every X steps
    save_steps = 200,               # saves every 100 training steps
    save_total_limit = 3,           # keeps latest 3 checkpoint
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard",
)


# 7) Define the trainer

# Suppose your dataset is tokenized like this
def tokenize_function(examples):
    # Combine instruction + input
    full_texts = [f"{ins}\n{inp}" if inp else ins for ins, inp in zip(examples["instruction"], examples["input"])]

    # Add the response
    full_texts = [f"{p}\n{resp}" for p, resp in zip(full_texts, examples["response"])]

    # Tokenize
    return tokenizer(
        full_texts,
        truncation=True,
        padding="max_length",
        max_length=max_seq_length
    )

# Apply tokenization
tokenized_dataset = dataset.map(tokenize_function, batched=True)

trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    peft_config=peft_config,
    args=training_arguments,
)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/112 [00:00<?, ?B/s]

agricult_data.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/5916 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'response'],
    num_rows: 1000
})


...

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

### Model Fine-Tuning (SFT)

In [ ]:
# 8) Launch training
trainer.train()

In [ ]:
!pip install -U bitsandbytes

### 4-bit Quantization & Base Model Loading

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    device_map="auto",
    load_in_4bit=True,
)

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Pick a checkpoint folder you want to use (e.g., last one)
checkpoint_path = "./mistral_agri_results/checkpoint-600"  # change as needed

# Load PEFT weights from that checkpoint
model = PeftModel.from_pretrained(base_model, checkpoint_path)
model.eval()



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


ImportError: Using `bitsandbytes` 4-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`

### Merging LoRA Weights & Saving Standalone Model

In [ ]:
# Merge LoRA weights into base model to make standalone
standalone_model = model.merge_and_unload()

# Save merged model
save_path = "./mistral_agri_merged"
standalone_model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ Standalone model saved at {save_path}")


In [ ]:
!pip install -U bitsandbytes

In [ ]:
from huggingface_hub import login
login()  # Enter your HF token when prompted


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import HfApi
import os

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import HfApi
import os

# Paths
model_folder = "./mistral_agri_merged"
username = "chanystrange"
repo_name = "mistral-agri-merged_143"
repo_id = f"{username}/{repo_name}"

# --- Load tokenizer (model itself doesn't need full GPU memory to push) ---
tokenizer = AutoTokenizer.from_pretrained(model_folder)
tokenizer.push_to_hub(repo_id)

# --- Push model to Hub without loading entire model into GPU ---
# This avoids OutOfMemoryError
model = AutoModelForCausalLM.from_pretrained(
    model_folder,
    device_map="auto",  # safe even on T4
    load_in_4bit=True   # memory-efficient
)
model.push_to_hub(repo_id)

print(f"✅ Model and tokenizer successfully pushed to Hugging Face Hub!")
print(f"Link: https://huggingface.co/{repo_id}")


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmpmjz5nns9/tokenizer.model      : 100%|##########|  587kB /  587kB            

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
/usr/local/lib/python3.12/dist-packages/accelerate/utils/modeling.py:1582: UserWarning: Current model requires 32.0 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmpy4jgfbrc/model.safetensors    :   1%|          | 29.4MB / 4.46GB            

✅ Model and tokenizer successfully pushed to Hugging Face Hub!
Link: https://huggingface.co/chanystrange/mistral-agri-merged_143


In [ ]:
import torch
torch.cuda.empty_cache()   # Clears cached memory
torch.cuda.reset_peak_memory_stats()


### Testing & Inference

In [ ]:
# 🔹 Imports
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
import torch

# --- Hugging Face repo ID of your pushed model ---
repo_id = "chanystrange/mistral-agri-merged_143"

# --- Clear GPU memory before loading ---
torch.cuda.empty_cache()

# --- 4-bit Quantization Config (memory-efficient) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",      # recommended for LLMs
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# --- Load tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(repo_id)
tokenizer.pad_token = tokenizer.eos_token

# --- Load model in 4-bit with device_map="auto" ---
model = AutoModelForCausalLM.from_pretrained(
    repo_id,
    device_map="auto",
    quantization_config=bnb_config
)

# --- Create text-generation pipeline ---
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7
)




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/4.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Device set to use cpu


In [ ]:
# --- Test prompt ---
prompt = "Explain the advantages of crop rotation."
output = generator(prompt)[0]['generated_text']

print("✅ Generated text:\n", output)

✅ Generated text:
 Explain the advantages of crop rotation.
Crop rotation is a practice in which different types of crops are grown in sequential order to enhance soil fertility, reduce pest and disease pressure, and promote biodiversity. The advantages of crop rotation include:

1. Soil Fertility Enhancement: Crop rotation involves planting crops with different root structures and nutrient requirements, which helps maintain soil structure and nutrient balance.

2. Pest and Disease Management: Rotating crops helps disrupt the life cycles of pests and diseases, reducing their buildup and minimizing damage to subsequent crops.

3. Weed Control: Incorporating cover crops, such as legumes, into the rotation can help control weeds by competing with them for resources and improving soil structure.

4. Biodiversity Promotion: Crop rotation supports a diverse community of plants, animals, and microorganisms, promoting ecosystem health and resilience.




In [ ]:
from transformers import pipeline

# Assuming 'model' and 'tokenizer' are already loaded
qa_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,   # adjust response length
    do_sample=True,
    temperature=0.7
)

# --- Ask a question ---
question = "What are the advantages of crop rotation in farming?"
response = qa_pipeline(question)[0]['generated_text']

print("✅ Response:\n", response)


Device set to use cuda:0


✅ Response:
 What are the advantages of crop rotation in farming?
Crop rotation is a practice in farming where different crops are grown in a sequential order on the same piece of land to improve soil fertility, prevent pest and disease buildup, and maintain overall crop health. Here are some benefits of crop rotation in farming:

1. Soil Health: Crop rotation helps to maintain soil health by allowing the depletion of certain nutrients by one crop to be replenished by another. For example, leguminous crops like beans and peas can fix nitrogen in the soil, while cereal crops like wheat and barley are nitrogen-hungry and benefit from the nitrogen fixed by legumes.

2. Prevention of Pest and Disease Buildup: By changing the crops grown in a particular area, you can disrupt the life cycles of pests and diseases that have adapted to specific crops. This can help to reduce pest and disease pressure and prevent their buildup.

3.
